# NPTEL Lecture Retrieval — Index Builder

Builds FAISS dense + BM25 sparse indexes for all chunking strategies on GPU.

**Before running:**
1. Settings → Accelerator → **GPU T4 x2** (or P100)
2. Settings → Internet → **On**
3. Add Data → your **nptel-lecture-segments** dataset
4. **Run All**

**Output:** `nptel_indexes.zip` in the Output panel → extract into local `data/indexes/`

In [ ]:
# ── Install ONLY the packages Kaggle does not pre-install ─────────────────────
# DO NOT install torch or sentence-transformers here.
# Kaggle has them pre-installed with the correct CUDA build.
# Reinstalling them breaks the CUDA kernel compatibility.
!pip install faiss-cpu rank-bm25 -q
print('Done.')

In [ ]:
# ── Verify all imports ────────────────────────────────────────────────────────
import importlib, importlib.util
importlib.invalidate_caches()

for mod in ['faiss', 'rank_bm25', 'sentence_transformers', 'torch']:
    spec = importlib.util.find_spec(mod)
    print(f'  {mod:<25} {"OK" if spec else "MISSING — pip install needed"}')

import torch
print(f'\ntorch version  : {torch.__version__}')
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ── Locate processed segments ─────────────────────────────────────────────────
import os, json, pickle, re, time, shutil
from pathlib import Path

KAGGLE_INPUT = Path('/kaggle/input')
OUTPUT_DIR   = Path('/kaggle/working/indexes')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED = None
for candidate in KAGGLE_INPUT.rglob('segments_c3.jsonl'):
    PROCESSED = candidate.parent
    print(f'Found segments in: {PROCESSED}')
    break

if PROCESSED is None:
    raise FileNotFoundError(
        'segments_c3.jsonl not found under /kaggle/input.\n'
        'Make sure you added the nptel-lecture-segments dataset via Add Data.'
    )

print('\nSegment files:')
for f in sorted(PROCESSED.glob('segments_*.jsonl')):
    print(f'  {f.name:<35} {f.stat().st_size/1e6:>6.1f} MB')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
EMBEDDING_MODEL = 'BAAI/bge-large-en-v1.5'
DEVICE          = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE      = 256 if DEVICE == 'cuda' else 64

ALL_STRATEGIES = {
    'c1':      'segments_c1.jsonl',
    'c2':      'segments_c2.jsonl',
    'c3':      'segments_c3.jsonl',
    'c2_w150': 'segments_c2_w150.jsonl',
    'c2_w250': 'segments_c2_w250.jsonl',
    'c3_t025': 'segments_c3_t025.jsonl',
    'c3_t040': 'segments_c3_t040.jsonl',
}

print(f'Device    : {DEVICE}')
print(f'Model     : {EMBEDDING_MODEL}')
print(f'Batch     : {BATCH_SIZE}')
print(f'Strategies: {list(ALL_STRATEGIES.keys())}')
print(f'Output    : {OUTPUT_DIR}')

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────────
_STOPWORDS = {
    'the','a','an','and','or','but','in','on','at','to','of',
    'is','it','as','be','by','we','so','do','if','he','she',
    'this','that','with','from','are','was','were','has','have',
    'had','not','also','will','can','its','our','their','what',
    'which','when','there','then','they','them','been','more',
    'into','than','just','some','would','about','because','now',
    'very','here','like','okay','right','yeah','uh','um',
}

def _tokenise(text):
    text = text.lower()
    text = re.sub(r'[^\w\s\-]', ' ', text)
    return [t for t in text.split() if len(t) >= 2 and t not in _STOPWORDS]

def load_segments(path):
    segs = []
    with open(path, encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if line:
                try: segs.append(json.loads(line))
                except: pass
    return segs

def build_passage_text(seg):
    parts = ['passage:']
    if seg.get('course_name'):   parts.append(f"[COURSE]: {seg['course_name']}")
    if seg.get('lecture_title'): parts.append(f"[LECTURE]: {seg['lecture_title']}")
    if seg.get('content_type'):  parts.append(f"[TYPE]: {seg['content_type']}")
    ocr = seg.get('ocr_text', '').strip()
    if ocr and not seg.get('ocr_failed', False):
        ocr_c = ocr.replace('\n---\n', ' ').replace('\n', ' ').strip()
        if len(ocr_c) > 10: parts.append(f'[SLIDE]: {ocr_c}')
    tx = seg.get('transcript', '').strip()
    if tx: parts.append(f'[SPEECH]: {tx}')
    return ' '.join(parts)

def build_bm25_text(seg):
    parts = []
    if seg.get('course_name'): parts.append(seg['course_name'])
    ocr = seg.get('ocr_text', '').strip()
    if ocr and not seg.get('ocr_failed', False):
        ocr_c = ocr.replace('\n---\n', ' ').replace('\n', ' ').strip()
        if len(ocr_c) > 10:
            parts.append(ocr_c)
            parts.append(ocr_c)  # x2 weight for slide text
    tx = seg.get('transcript', '').strip()
    if tx: parts.append(tx)
    return ' '.join(parts)

print('Helpers ready ✅')

In [ ]:
# ── Build BM25 indexes (CPU, ~30 seconds total) ───────────────────────────────
from rank_bm25 import BM25Okapi

t_total = time.time()
for strategy, fname in ALL_STRATEGIES.items():
    out_path   = OUTPUT_DIR / f'bm25_{strategy}.pkl'
    jsonl_path = PROCESSED / fname
    if not jsonl_path.exists():
        print(f'  [{strategy:<10}] SKIP — {fname} not found')
        continue
    print(f'  [{strategy:<10}] BM25 ...', end=' ', flush=True)
    t0       = time.time()
    segments = load_segments(jsonl_path)
    corpus   = [_tokenise(build_bm25_text(s)) for s in segments]
    bm25     = BM25Okapi(corpus)
    with open(out_path, 'wb') as fh:
        pickle.dump({'bm25': bm25, 'corpus': corpus}, fh, protocol=pickle.HIGHEST_PROTOCOL)
    print(f'{len(segments):,} segs | vocab={len(bm25.idf):,} | '
          f'{time.time()-t0:.1f}s | {out_path.stat().st_size/1e6:.0f} MB  ✅')

print(f'\n✅ BM25 done in {time.time()-t_total:.1f}s')

In [ ]:
# ── Load embedding model ──────────────────────────────────────────────────────
# sentence-transformers is pre-installed on Kaggle — do NOT reinstall it
from sentence_transformers import SentenceTransformer

print(f'Loading {EMBEDDING_MODEL} on {DEVICE} ...')
t0 = time.time()
embed_model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
dim = embed_model.get_sentence_embedding_dimension()
print(f'Loaded in {time.time()-t0:.1f}s | dim = {dim}')

# Quick smoke test to confirm GPU kernels work before running the full loop
print('Running smoke test ...', end=' ', flush=True)
test_emb = embed_model.encode(['test sentence'], convert_to_numpy=True)
print(f'OK — output shape {test_emb.shape} ✅')

In [ ]:
# ── Build FAISS dense indexes ─────────────────────────────────────────────────
import faiss
import numpy as np

META_FIELDS = [
    'segment_id', 'course_id', 'course_name', 'instructor', 'institute',
    'lecture_title', 'lecture_number', 'youtube_url', 'youtube_deep_link',
    'start_sec', 'end_sec', 'duration_sec',
    'transcript', 'ocr_text', 'ocr_confidence', 'ocr_failed',
    'is_code_segment', 'content_type', 'chunking_strategy',
    'word_count', 'boundary_method',
]

t_total = time.time()
for strategy, fname in ALL_STRATEGIES.items():
    faiss_out  = OUTPUT_DIR / f'faiss_{strategy}.index'
    meta_out   = OUTPUT_DIR / f'metadata_{strategy}.json'
    jsonl_path = PROCESSED / fname

    if not jsonl_path.exists():
        print(f'\n[{strategy}] SKIP — {fname} not found')
        continue

    print(f'\n[{strategy}] {fname}', flush=True)
    segments = load_segments(jsonl_path)
    texts    = [build_passage_text(s) for s in segments]
    print(f'  {len(segments):,} segments loaded', flush=True)

    print(f'  Embedding (batch={BATCH_SIZE}) ...', flush=True)
    t0 = time.time()
    embeddings = embed_model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype('float32')
    elapsed = time.time() - t0
    print(f'  {len(texts):,} segs in {elapsed:.1f}s ({len(texts)/elapsed:.0f} segs/sec)', flush=True)

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    faiss.write_index(index, str(faiss_out))
    print(f'  Saved {faiss_out.name}  ({faiss_out.stat().st_size/1e6:.0f} MB)', flush=True)

    slim = [{k: s.get(k) for k in META_FIELDS} for s in segments]
    meta_out.write_text(json.dumps(slim, ensure_ascii=False), encoding='utf-8')
    print(f'  Saved {meta_out.name}  ({meta_out.stat().st_size/1e6:.0f} MB)', flush=True)

    del embeddings
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

print(f'\n✅ FAISS done in {time.time()-t_total:.1f}s')

In [ ]:
# ── Summary + zip ─────────────────────────────────────────────────────────────
print('='*65)
print(f'{"Strategy":<12} {"FAISS":<15} {"BM25":<15} {"Metadata"}')
print('-'*65)
total_mb = 0
all_ok   = True
for strategy in ALL_STRATEGIES:
    fi = OUTPUT_DIR / f'faiss_{strategy}.index'
    bi = OUTPUT_DIR / f'bm25_{strategy}.pkl'
    mi = OUTPUT_DIR / f'metadata_{strategy}.json'
    fi_s = f'{fi.stat().st_size/1e6:.0f}MB ✅' if fi.exists() else 'MISSING ❌'
    bi_s = f'{bi.stat().st_size/1e6:.0f}MB ✅' if bi.exists() else 'MISSING ❌'
    mi_s = f'{mi.stat().st_size/1e6:.0f}MB ✅' if mi.exists() else 'MISSING ❌'
    for p in [fi, bi, mi]:
        if p.exists(): total_mb += p.stat().st_size/1e6
        else: all_ok = False
    print(f'{strategy:<12} {fi_s:<15} {bi_s:<15} {mi_s}')
print('='*65)
print(f'Total: {total_mb:.0f} MB | {"ALL COMPLETE ✅" if all_ok else "SOME MISSING ❌"}')

zip_base = '/kaggle/working/nptel_indexes'
print(f'\nZipping → {zip_base}.zip ...')
shutil.make_archive(zip_base, 'zip', OUTPUT_DIR)
print(f'✅ nptel_indexes.zip ready ({Path(zip_base+".zip").stat().st_size/1e6:.0f} MB)')
print('\nDownload from Output panel, then locally:')
print('  unzip nptel_indexes.zip -d data/indexes/')